# Практика: Структурированный вывод с GigaChat API

В этом ноутбуке мы научимся получать от LLM не просто текст, а **структурированные данные** в формате JSON, которые можно сразу использовать в коде.

**Финальное задание:** построить мини-пайплайн для автоматического разбора резюме — программа будет извлекать из произвольного текста структурированный профиль кандидата.

## 0. Получение API-ключа GigaChat

**Пошаговая инструкция:**

1. Перейдите на **[сайт](https://developers.sber.ru/)** и зарегистрируйтесь
2. В правом меню выберите раздел **"API Keys"**
3. Нажмите **"Create API Key"**, введите название (например, `GIGA_API_KEY`) и нажмите **"Create"**
4. Скопируйте ключ —  `MDF...`

> **Важно:** ключ показывается только один раз. Сохраните его сразу.
>
> Новые аккаунты получают бесплатные токены для тестирования.

Создайте файл `.env` в той же папке, что и этот ноутбук:

```
GIGA_API_KEY=gsk-ВАШ_КЛЮЧ_ЗДЕСЬ
```

## 1. Установка и настройка клиента

In [17]:
!pip install gigachat python-dotenv

import os
import json
import time
import requests
import uuid
import urllib3
from dotenv import load_dotenv
from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole

load_dotenv()

api_key = os.getenv('GIGA_API_KEY')
client = GigaChat(credentials=api_key, verify_ssl_certs=False, model="GigaChat-2")

print(f'Ключ GIGACHAT загружен: {api_key[:5] if api_key else "Не найден"}...')

Defaulting to user installation because normal site-packages is not writeable
Ключ GIGACHAT загружен: MDFhM...


In [18]:
response = client.chat("Привет! Как дела?")
print(response.choices[0].message.content)

Привет! Всё отлично, спасибо. Готов обсудить что-нибудь интересное или помочь разобраться с задачей. Чем займёмся?


## 2. Проблема: LLM возвращает неструктурированный текст

Попросим модель извлечь данные из текста обычным запросом — и посмотрим, почему такой вывод неудобен для программы.

In [20]:
resume_text = """
Привет! Меня зовут Анна Соколова, мне 28 лет. Последние 4 года работаю
бэкенд-разработчиком в финтех-компании. Основной стек: Python, FastAPI,
PostgreSQL, Redis. Есть опыт с Docker и Kubernetes. Закончила МГТУ им. Баумана
по специальности «Прикладная математика». Ищу позицию Senior Python Developer
с зарплатой от 300 000 рублей. Готова к удалённой работе или гибриду.
"""
response = client.chat.create(
    {
        "model": "GigaChat-2",
        "messages": [
            {
                "role": "user",
                "content": f'Извлеки основную информацию из резюме:\n{resume_text}'
            }
        ]
    }
)


raw_output = response.messages[0].content[0].text
print('Вывод модели:')
print(raw_output)

# Обычный запрос без требования структуры
# response = client.chat.completions.create(
#     model="GigaChat-2",
#     messages=[
#         {'role': 'user', 'content': f'Извлеки основную информацию из резюме:\n{resume_text}'}
#     ]
# )

# raw_output = response.choices[0].message.content
# print('Вывод модели:')
# print(raw_output)
# print()

# Попытка использовать вывод в коде — неудача
print('Попытка получить зарплату из ответа:')
try:
    data = json.loads(raw_output)
    print(data['salary'])
except Exception as e:
    print(f'Ошибка: {e}')
    print('Неструктурированный текст нельзя напрямую использовать в коде!')

Вывод модели:
**Основная информация из резюме Анны Соколовой:**

- **Имя:** Анна Соколова  
- **Возраст:** 28 лет  
- **Опыт работы:** 4 года бэкенд-разработчиком  
- **Место работы:** Финтех-компания  
- **Ключевые технологии:**  
  - Python  
  - FastAPI  
  - PostgreSQL  
  - Redis  
  - Docker  
  - Kubernetes  
- **Образование:**  
  - МГТУ им. Баумана, специальность «Прикладная математика»  
- **Желаемая позиция:** Senior Python Developer  
- **Зарплатные ожидания:** от 300 000 руб.  
- **Готова к формату работы:** удалённая работа или гибридный график
Попытка получить зарплату из ответа:
Ошибка: Expecting value: line 1 column 1 (char 0)
Неструктурированный текст нельзя напрямую использовать в коде!


## 3. Решение: JSON-режим

Передадим параметр `response_format={'type': 'json_object'}` — и модель вернёт валидный JSON.

> **Правило:** слово `JSON` должно присутствовать в системном промпте — иначе API вернёт ошибку.

In [24]:
system_prompt = """
Ты — HR-ассистент. Извлекай данные из резюме и возвращай строго в формате JSON.
Схема ответа:
{
  "name": string,
  "age": number или null,
  "experience_years": number,
  "current_position": string,
  "skills": [string],
  "education": string или null,
  "target_position": string,
  "salary_from": number или null,
  "work_format": string
}
"""

schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": ["number", "null"]},
        "experience_years": {"type": "number"},
        "current_position": {"type": "string"},
        "skills": {"type": "array", "items": {"type": "string"}},
        "education": {"type": ["string", "null"]},
        "target_position": {"type": "string"},
        "salary_from": {"type": ["number", "null"]},
        "work_format": {"type": "string"}
    },
    "required": ["name", "experience_years", "current_position", "skills", "target_position", "work_format"]
}


response = client.chat.create(
    {
        "model": "GigaChat-2",
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Обработай резюме:\n{resume_text}"}
        ],
        "response_format": {
            "type": "json_schema",
            "schema": schema
        }
    }
)


raw_output = response.messages[0].content[0].text
candidate = json.loads(raw_output)


print('Структурированный профиль кандидата:')
print(json.dumps(candidate, ensure_ascii=False, indent=2))
print()

# Теперь данные легко использовать в коде
print(f"Имя: {candidate['name']}")
print(f"Опыт: {candidate['experience_years']} лет")
print(f"Навыки: {', '.join(candidate['skills'])}")
print(f"Ожидаемая зарплата: {candidate['salary_from']:,} руб.".replace(',', ' '))

Структурированный профиль кандидата:
{
  "name": "Анна Соколова",
  "age": 28,
  "experience_years": 4,
  "current_position": "бэкенд-разработчик",
  "skills": [
    "Python",
    "FastAPI",
    "PostgreSQL",
    "Redis",
    "Docker",
    "Kubernetes"
  ],
  "education": "МГТУ им. Баумана, специальность «Прикладная математика»",
  "target_position": "Senior Python Developer",
  "salary_from": 300000,
  "work_format": "удалённая работа или гибрид"
}

Имя: Анна Соколова
Опыт: 4 лет
Навыки: Python, FastAPI, PostgreSQL, Redis, Docker, Kubernetes
Ожидаемая зарплата: 300 000 руб.


## 4. Пайплайн: обработка нескольких резюме

Завернём логику в функцию и запустим её на нескольких резюме сразу.

In [28]:
from gigachat import GigaChat
from gigachat.exceptions import RateLimitError, GigaChatException

def parse_resume(text: str) -> dict | None:
    """Извлекает структурированные данные из текста резюме."""
    system_prompt = """
    Ты — HR-ассистент. Извлекай данные из резюме и возвращай строго в формате JSON.
    Схема: {"name": string, "experience_years": number, "skills": [string],
    "target_position": string, "salary_from": number или null, "work_format": string}
    """
    schema = {
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "experience_years": {"type": "number"},
            "skills": {"type": "array", "items": {"type": "string"}},
            "target_position": {"type": "string"},
            "salary_from": {"type": ["number", "null"]},
            "work_format": {"type": "string"}
        },
        "required": ["name", "experience_years", "skills", "target_position", "salary_from", "work_format"]
    }

    try:
        response = client.chat.create(
            {
                "model": "GigaChat-2",
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Обработай резюме:\n{text}"}
                ],
                "response_format": {
                    "type": "json_schema",
                    "schema": schema
                }
            }
        )
    
        return json.loads(response.messages[0].content[0].text)
    
    except RateLimitError as e:
        print(f'Лимит запросов. Ждём 5 сек... ({e})')
        time.sleep(5)
        return None
    except Exception as e:
        print(f'Ошибка: {e}')
        return None    
    


resumes = [
    """
    Иван Петров, 32 года. 7 лет в разработке игр на Unity и C#.
    Делал мобильные игры для iOS/Android, есть опыт с Blender.
    Ищу Lead Game Developer, готов к офису в Москве. Зарплата от 250 000.
    """,
    """
    Мария Иванова. Data Scientist с опытом 3 года.
    Стек: Python, pandas, scikit-learn, PyTorch, SQL.
    Работала в e-commerce над рекомендательными системами.
    Хочу ML Engineer, только удалёнка, зарплата не принципиальна.
    """,
    """
    Дмитрий Козлов, 25 лет, junior frontend. Знаю React, TypeScript, CSS.
    Полгода коммерческого опыта в веб-студии. Ищу работу в продуктовой компании.
    Рассматриваю от 120 000, гибрид или удалёнка.
    """,
    """
    Екатерина Смирнова, 29 лет. DevOps-инженер, 5 лет опыта.
    Умею: Kubernetes, Docker, Terraform, AWS, CI/CD (GitLab, GitHub Actions).
    Работала в банковском секторе, есть опыт с highload-системами.
    Ищу Senior DevOps, удалёнка обязательно. Зарплата от 300 000.
    """,
    """
    Алексей Новиков, 35 лет. Fullstack-разработчик, 10 лет опыта.
    Стек: Python (Django, FastAPI), Vue.js, PostgreSQL, Redis, Docker.
    Последние 3 года — тимлид команды из 6 человек в SaaS-компании.
    Рассматриваю роль Engineering Manager или Staff Engineer. От 400 000, гибрид.
    """,
    """
    Юлия Белова. QA Engineer, 4 года в тестировании.
    Автотесты на Python (pytest, Selenium, Playwright), API-тестирование (Postman, requests).
    Опыт в agile-командах, знакома с нагрузочным тестированием (Locust).
    Ищу Middle/Senior QA, гибрид, зарплата от 180 000.
    """,
    """
    Сергей Орлов, 27 лет. Android-разработчик, 3 года опыта.
    Kotlin, Jetpack Compose, Room, Retrofit, Coroutines. Публиковал 2 приложения в Google Play.
    Немного знаю iOS (Swift). Ищу Middle Android Developer в продуктовую компанию.
    Только удалёнка. Зарплата от 210 000.
    """
]

results = []
for i, resume in enumerate(resumes, 1):
    print(f'Обрабатываю резюме {i}/{len(resumes)}...')
    parsed = parse_resume(resume)
    if parsed:
        results.append(parsed)

print(f'\nОбработано: {len(results)} из {len(resumes)} резюме')
print()
for r in results:
    salary = f"{r['salary_from']:,}".replace(',', ' ') + ' руб.' if r.get('salary_from') else 'не указана'
    print(f"• {r['name']} | {r['target_position']} | опыт {r['experience_years']} л. | зарплата: {salary}")


Обрабатываю резюме 1/7...
Обрабатываю резюме 2/7...
Обрабатываю резюме 3/7...
Обрабатываю резюме 4/7...
Обрабатываю резюме 5/7...
Обрабатываю резюме 6/7...
Обрабатываю резюме 7/7...

Обработано: 7 из 7 резюме

• Иван Петров | Lead Game Developer | опыт 7 л. | зарплата: 250 000 руб.
• Мария Иванова | ML Engineer | опыт 3 л. | зарплата: не указана
• Дмитрий Козлов | junior frontend | опыт 0.5 л. | зарплата: 120 000 руб.
• Екатерина Смирнова | Senior DevOps | опыт 5 л. | зарплата: 300 000 руб.
• Алексей Новиков | Engineering Manager или Staff Engineer | опыт 10 л. | зарплата: 400 000 руб.
• Юлия Белова | Middle/Senior QA | опыт 4 л. | зарплата: 180 000 руб.
• Сергей Орлов | Middle Android Developer | опыт 3 л. | зарплата: 210 000 руб.


## 5. Фильтрация и анализ собранных данных

Структурированные данные позволяют легко фильтровать кандидатов без дополнительных запросов к модели.

Ниже показан один пример фильтрации — изучите его, затем **выполните самостоятельные задания** в следующей ячейке.


In [29]:
# Пример: кандидаты с опытом 3+ лет и зарплатой до 300 000
filtered = [
    r for r in results
    if r['experience_years'] >= 3
    and (r['salary_from'] is None or r['salary_from'] <= 300_000)
]

print(f'Кандидаты (опыт ≥ 3 лет, зарплата ≤ 300 000): {len(filtered)}')
for r in filtered:
    print(f"  → {r['name']}: {', '.join(r['skills'][:3])}...")

print()

# Самый опытный кандидат
if results:
    top = max(results, key=lambda r: r['experience_years'])
    print(f"Самый опытный: {top['name']} ({top['experience_years']} лет опыта)")

# Все упомянутые навыки
all_skills = [skill for r in results for skill in r['skills']]
unique_skills = sorted(set(all_skills))
print(f'\nВсе навыки в пуле: {len(unique_skills)}')
print(', '.join(unique_skills))


Кандидаты (опыт ≥ 3 лет, зарплата ≤ 300 000): 5
  → Иван Петров: разработка игр на Unity, разработка игр на C#, опыт с Blender...
  → Мария Иванова: Python, pandas, scikit-learn...
  → Екатерина Смирнова: Kubernetes, Docker, Terraform...
  → Юлия Белова: Автотесты на Python (pytest, Selenium, Playwright), API-тестирование (Postman, requests), Нагрузочное тестирование (Locust)...
  → Сергей Орлов: Kotlin, Jetpack Compose, Room...

Самый опытный: Алексей Новиков (10 лет опыта)

Все навыки в пуле: 28
API-тестирование (Postman, requests), AWS, CI/CD (GitLab, GitHub Actions), CSS, Coroutines, Docker, Jetpack Compose, Kotlin, Kubernetes, PostgreSQL, PyTorch, Python, Python (Django, FastAPI), React, Redis, Retrofit, Room, SQL, Terraform, TypeScript, Vue.js, pandas, scikit-learn, Автотесты на Python (pytest, Selenium, Playwright), Нагрузочное тестирование (Locust), опыт с Blender, разработка игр на C#, разработка игр на Unity


### Задание: фильтрация кандидатов

Используя список `results`, выполните следующие задачи в ячейке ниже:

1. **Удалёнщики-синьоры** — найдите кандидатов, у которых `work_format` содержит слово `удалёнка` и `experience_years >= 4`.
2. **Бюджетный найм** — выведите всех кандидатов с зарплатными ожиданиями не выше 230 000 руб. (или без указанной зарплаты).
3. **Python-разработчики** — отберите кандидатов, у которых в `skills` есть `Python`.
4. **Топ-3 по опыту** — выведите трёх самых опытных кандидатов в порядке убывания стажа.
5. **Статистика зарплат** — посчитайте среднюю, минимальную и максимальную ожидаемую зарплату среди тех, кто её указал.


In [35]:
# Задание 1: удалёнщики с опытом 4+ лет
# TODO: напишите list comprehension или цикл
remote_seniors = [
    r for r in results
    if 'удалёнка' in r['work_format'].lower() and r['experience_years'] >= 4
]

print("1. Удалёнщики-синьоры (опыт ≥ 4, удалёнка):")
for r in remote_seniors:
    print(f"   → {r['name']} | {r['experience_years']} л. | {r['work_format']}")
print(f"   Итого: {len(remote_seniors)}")
print()

# Задание 2: зарплатные ожидания <= 230 000 (или не указаны)
# TODO:
budget_candidates = [
    r for r in results
    if r['salary_from'] is None or r['salary_from'] <= 230_000
]

print("2. Бюджетный найм (зарплата ≤ 230 000 или не указана):")
for r in budget_candidates:
    salary = f"{r['salary_from']:,}".replace(',', ' ') + ' руб.' if r['salary_from'] else 'не указана'
    print(f"   → {r['name']} | {salary}")
print(f"   Итого: {len(budget_candidates)}")
print()

# Задание 3: кандидаты с Python в навыках
# Подсказка: используйте any() или проверку 'Python' in r['skills']
# TODO:
python_devs = [r for r in results if 'Python' in r['skills']]

print("3. Python-разработчики:")
for r in python_devs:
    print(f"   → {r['name']} | {r['target_position']}")
print(f"   Итого: {len(python_devs)}")
print()

# Задание 4: топ-3 по опыту
# Подсказка: sorted(..., key=..., reverse=True)[:3]
# TODO:
top3 = sorted(results, key=lambda r: r['experience_years'], reverse=True)[:3]

print("4. Топ-3 по опыту:")
for place, r in enumerate(top3, 1):
    print(f"   {place}. {r['name']} — {r['experience_years']} лет")
print()

# Задание 5: статистика зарплат
# Подсказка: сначала соберите список salary_from тех, у кого он не None
# TODO:
salaries = [r['salary_from'] for r in results if r['salary_from'] is not None]

if salaries:
    avg_salary = sum(salaries) / len(salaries)
    min_salary = min(salaries)
    max_salary = max(salaries)

    print("5. Статистика зарплат (только с указанной зарплатой):")
    print(f"   Количество: {len(salaries)}")
    print(f"   Средняя:    {avg_salary:,.0f} руб.".replace(',', ' '))
    print(f"   Минимальная: {min_salary:,.0f} руб.".replace(',', ' '))
    print(f"   Максимальная: {max_salary:,.0f} руб.".replace(',', ' '))
else:
    print("5. Нет кандидатов с указанной зарплатой.")

1. Удалёнщики-синьоры (опыт ≥ 4, удалёнка):
   → Екатерина Смирнова | 5 л. | удалёнка
   Итого: 1

2. Бюджетный найм (зарплата ≤ 230 000 или не указана):
   → Мария Иванова | не указана
   → Дмитрий Козлов | 120 000 руб.
   → Юлия Белова | 180 000 руб.
   → Сергей Орлов | 210 000 руб.
   Итого: 4

3. Python-разработчики:
   → Мария Иванова | ML Engineer
   Итого: 1

4. Топ-3 по опыту:
   1. Алексей Новиков — 10 лет
   2. Иван Петров — 7 лет
   3. Екатерина Смирнова — 5 лет

5. Статистика зарплат (только с указанной зарплатой):
   Количество: 6
   Средняя:    243 333 руб.
   Минимальная: 120 000 руб.
   Максимальная: 400 000 руб.


## 6. Сравнение моделей

GigaChat предоставляет доступ к нескольким моделям с разными характеристиками. Сравним их по **скорости** и **качеству** извлечения структурированных данных.

| Модель | Размер | Типичная скорость |
|---|---|---|
| `GigaChat-2` | - | ~- токенов/с |
| `GigaChat-2-Pro` | - | ~- токенов/с |
| `GigaChat-2-Max` | - | ~- токенов/с |
| `GigaChat-3-Ultra` | - | ~- токенов/с |

> 💡 Актуальный список моделей всегда доступен на [сайте](https://developers.sber.ru/docs/ru/gigachat/guides/selecting-a-model#dostupnye-modeli)


In [40]:
import time

schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "experience_years": {"type": "number"},
        "skills": {"type": "array", "items": {"type": "string"}},
        "target_position": {"type": "string"},
        "salary_from": {"type": ["number", "null"]},
        "work_format": {"type": "string"}
    },
    "required": ["name", "experience_years", "skills", "target_position", "salary_from", "work_format"]
}

# Резюме для теста
test_resume = """
Андрей Захаров, 31 год. Backend-разработчик, 6 лет опыта.
Основной стек: Go, Python, gRPC, Kafka, PostgreSQL, Kubernetes.
Строил высоконагруженные сервисы в финтех и логистике.
Ищу Senior/Lead Backend Engineer. Удалёнка, зарплата от 380 000.
"""

system_prompt = """
Ты — HR-ассистент. Извлекай данные из резюме и возвращай строго в формате JSON.
Схема: {"name": string, "experience_years": number, "skills": [string],
"target_position": string, "salary_from": number или null, "work_format": string}
"""

models_to_compare = [
    'GigaChat-2',
    'GigaChat-2-Pro',
    'GigaChat-2-Max',
    'GigaChat-3-Ultra',
]

comparison_results = []

for model_name in models_to_compare:
    try:
        start = time.time()
        response = client.chat.create(
            {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Обработай резюме:\n{test_resume}"}
                ],
                "response_format": {
                    "type": "json_schema",
                    "schema": schema
                }
            }
        )
        elapsed = time.time() - start

        raw_text = response.messages[0].content[0].text
        parsed = json.loads(raw_text)
        usage = response.usage
        total_tokens = usage.total_tokens

        comparison_results.append({
            'model': model_name,
            'time_sec': round(elapsed, 2),
            'total_tokens': total_tokens,
            'tokens_per_sec': round(total_tokens / elapsed) if elapsed > 0 else 0,
            'skills_found': len(parsed.get('skills', [])),
            'parsed': parsed,
        })
        print(f'OK {model_name}: {elapsed:.2f} с, {usage.total_tokens} токенов')
        time.sleep(1)  # пауза между запросами
    except Exception as e:
        print(f'ОШИБКА {model_name}: {e}')

print()
header = f"{'Модель':<35} {'Время':>8} {'Ток/с':>8} {'Навыков':>9}"
print(header)
print('-' * 65)
for r in comparison_results:
    row = f"{r['model']:<35} {r['time_sec']:>7.2f}с {r['tokens_per_sec']:>8} {r['skills_found']:>9}"
    print(row)


OK GigaChat-2: 1.21 с, 174 токенов
OK GigaChat-2-Pro: 2.87 с, 172 токенов
OK GigaChat-2-Max: 3.04 с, 269 токенов
OK GigaChat-3-Ultra: 2.94 с, 268 токенов

Модель                                 Время    Ток/с   Навыков
-----------------------------------------------------------------
GigaChat-2                             1.21с      144         6
GigaChat-2-Pro                         2.87с       60         6
GigaChat-2-Max                         3.04с       89         6
GigaChat-3-Ultra                       2.94с       91         6


### Задание: анализ результатов сравнения

После выполнения ячейки выше ответьте на вопросы (в новой ячейке типа Markdown или в комментариях):

1. Какая модель оказалась **самой быстрой** по времени ответа? А по числу токенов в секунду?
2. Все ли модели извлекли одинаковое количество навыков? Есть ли расхождения?
3. Когда стоит предпочесть большую модель (70B) маленькой (8B)? Приведите пример задачи.



##Ответы:
1. по времени ответа и по числу токенов в секунду самая быстрая - GigaChat-2
2. Да, все четыре модели извлекли ровно 6 навыков.
3. Большую модель стоит брать, когда задача требует рассуждения, а не просто извлечения. Например нужен анализ, подходит ли кандидат на вакансию и почемму.

---
## Самостоятельное задание: парсер вакансий

Вы видели, как разбирать **резюме**. Теперь сделайте то же самое для **вакансий**.

### Задача
Реализуйте функцию `parse_vacancy(text: str) -> dict | None`, которая принимает текст вакансии и возвращает структурированный JSON со следующими полями:

```json
{
  "position": "название должности",
  "company": "название компании или null",
  "required_skills": ["список обязательных навыков"],
  "nice_to_have": ["список желательных навыков"],
  "experience_required": "например '3-5 лет' или null",
  "salary_to": "верхняя граница зарплаты или null",
  "work_format": "офис / удалёнка / гибрид"
}
```

### Требования
- Используйте `response_format={'type': 'json_object'}`
- Системный промпт должен содержать слово `JSON` и описание схемы
- Оберните вызов в `try/except`
- Проверьте функцию на всех пяти тестовых вакансиях ниже

### Бонус ⭐
Напишите функцию `match_candidates(vacancy: dict, candidates: list) -> list`, которая по уже разобранным данным подбирает подходящих кандидатов из списка `results` (совпадение хотя бы 2 навыков из `required_skills`).

In [41]:
# Тестовые вакансии
vacancy_1 = """
Компания TechFlow ищет Senior Python Developer.
Обязательно: Python 5+ лет, FastAPI или Django, PostgreSQL, Docker.
Желательно: Kubernetes, опыт с микросервисами, Redis.
Условия: гибрид (Москва), зарплата до 400 000 рублей, ДМС.
"""

vacancy_2 = """
Ищем Middle Data Scientist в e-commerce стартап.
Требования: Python, pandas, scikit-learn, опыт построения ML-моделей от 2 лет.
Плюсом будет: PyTorch, опыт с рекомендательными системами, A/B-тесты.
Полная удалёнка, зарплата до 220 000. Команда 5 человек.
"""

vacancy_3 = """
Frontend Developer (React) — продуктовая компания в сфере EdTech.
Must have: React, TypeScript, REST API, опыт от 1 года.
Nice to have: Next.js, тестирование (Jest), Storybook.
Офис в СПб или удалёнка, вилка 150 000 – 200 000.
"""


def parse_vacancy(text: str) -> dict | None:
    """
    Извлекает структурированные данные из текста вакансии.

    Args:
        text: Текст вакансии в произвольном формате

    Returns:
        Словарь с полями position, company, required_skills,
        nice_to_have, experience_required, salary_to, work_format
        или None при ошибке
    """

    system_prompt = """
    Ты — HR-ассистент. Извлекай данные из вакансии и возвращай строго в формате JSON.
    Схема ответа:
    {
      "position": string,
      "company": string или null,
      "required_skills": [string],
      "nice_to_have": [string],
      "experience_required": string или null,
      "salary_to": number или null,
      "work_format": string
    }
    """
    schema = {
        "type": "object",
        "properties": {
            "position": {"type": "string"},
            "company": {"type": ["string", "null"]},
            "required_skills": {"type": "array", "items": {"type": "string"}},
            "nice_to_have": {"type": "array", "items": {"type": "string"}},
            "experience_required": {"type": ["string", "null"]},
            "salary_to": {"type": ["number", "null"]},
            "work_format": {"type": "string"}
        },
        "required": ["position", "company", "required_skills", "nice_to_have",
                     "experience_required", "salary_to", "work_format"]
    }

    try:
        response = client.chat.create(
            {
                "model": "GigaChat-2",
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Обработай вакансию:\n{text}"}
                ],
                # По заданию было бы: response_format={'type': 'json_object'}
                # Но GigaChat его не поддерживает — используем json_schema
                "response_format": {
                    "type": "json_schema",
                    "schema": schema
                }
            }
        )
        raw_text = response.messages[0].content[0].text
        return json.loads(raw_text)
    except RateLimitError as e:
        print(f'Лимит запросов. Ждём 5 сек... ({e})')
        time.sleep(5)
        return None
    except Exception as e:
        print(f'Ошибка: {e}')
        return None

    # TODO: Реализуйте функцию
    # Подсказки:
    # 1. Системный промпт: опишите роль и схему JSON
    # 2. Не забудьте слово JSON в системном промпте!
    # 3. Передайте text в user-сообщение
    # 4. Используйте response_format={'type': 'json_object'}
    # 5. Верните json.loads(response.choices[0].message.content)

    pass  # Удалите эту строку и напишите свой код


# Тестирование
for i, vacancy in enumerate([vacancy_1, vacancy_2, vacancy_3], 1):
    print(f'{'=' * 55}')
    print(f'Вакансия {i}')
    print('=' * 55)
    result = parse_vacancy(vacancy)
    if result:
        print(json.dumps(result, ensure_ascii=False, indent=2))
    print()

Вакансия 1
{
  "position": "Senior Python Developer",
  "company": "TechFlow",
  "required_skills": [
    "Python 5+ лет",
    "FastAPI или Django",
    "PostgreSQL",
    "Docker"
  ],
  "nice_to_have": [
    "Kubernetes",
    "опыт с микросервисами",
    "Redis"
  ],
  "experience_required": "5+ лет",
  "salary_to": 400000,
  "work_format": "гибрид (Москва)"
}

Вакансия 2
{
  "position": "Middle Data Scientist",
  "company": "e-commerce стартап",
  "required_skills": [
    "Пython",
    "pandas",
    "scikit-learn",
    "опыт построения ML-моделей"
  ],
  "nice_to_have": [
    "PyTorch",
    "опыт с рекомендательными системами",
    "A/B-тесты"
  ],
  "experience_required": "опыт построения ML-моделей от 2 лет",
  "salary_to": 220000,
  "work_format": "полная удалёнка"
}

Вакансия 3
{
  "position": "Frontend Developer (React)",
  "company": "продуктовая компания в сфере EdTech",
  "required_skills": [
    "React",
    "TypeScript",
    "REST API"
  ],
  "nice_to_have": [
    "Next.js"

In [42]:
# Бонус: подбор кандидатов под вакансию

def match_candidates(vacancy: dict, candidates: list) -> list:
    """
    Подбирает кандидатов из списка по совпадению навыков.

    Args:
        vacancy: Разобранная вакансия (результат parse_vacancy)
        candidates: Список разобранных резюме (результаты parse_resume)

    Returns:
        Список подходящих кандидатов с указанием совпавших навыков
    """
    required = {s.lower() for s in vacancy.get("required_skills", [])}
    matched = []

    for candidate in candidates:
        candidate_skills = {s.lower() for s in candidate.get("skills", [])}
        common = required & candidate_skills

        if len(common) >= 2:
            matched.append({
                "name": candidate.get("name"),
                "target_position": candidate.get("target_position"),
                "experience_years": candidate.get("experience_years"),
                "matched_skills": sorted(common),
                "match_count": len(common),
            })

    # Сортируем по числу совпавших навыков (лучшие — первыми)
    matched.sort(key=lambda x: x["match_count"], reverse=True)
    return matched


# Пример использования бонуса
print("\n" + "=" * 55)
print("Бонус: подбор кандидатов под вакансию 1")
print("=" * 55)

vacancy_parsed = parse_vacancy(vacancy_1)
if vacancy_parsed:
    suitable = match_candidates(vacancy_parsed, results)
    for s in suitable:
        print(f"• {s['name']} | {s['target_position']} | "
              f"совпало {s['match_count']}: {', '.join(s['matched_skills'])}")
    
    # TODO: Реализуйте функцию
    # Подсказки:
    # 1. Возьмите required_skills из vacancy
    # 2. Для каждого кандидата найдите пересечение его skills с required_skills
    # 3. Верните кандидатов, у которых совпало >= 2 навыков
    # 4. В возвращаемый словарь добавьте поле matched_skills

    pass


Бонус: подбор кандидатов под вакансию 1
• Алексей Новиков | Engineering Manager или Staff Engineer | совпало 2: docker, postgresql


---
## Чек-лист перед сдачей

- [ ] Все ячейки выполняются без ошибок (Kernel → Restart & Run All)
- [ ] Раздел 5: выполнены все 5 заданий по фильтрации
- [ ] Раздел 6: таблица сравнения моделей заполнена, письменные ответы добавлены
- [ ] Функция `parse_vacancy()` реализована и возвращает корректный JSON для всех 5 вакансий
- [ ] Все 7 полей схемы присутствуют в ответе (или `null`, если данных нет)
- [ ] Используется `response_format={'type': 'json_object'}`
- [ ] В коде есть обработка ошибок `try/except`
- [ ] API-ключ не хранится в коде — только через `.env`
- [ ] (Бонус) Функция `match_candidates()` реализована
